In [2]:
!pip install torch torchvision torchaudio
!pip install transformers datasets accelerate trl pandas

In [3]:
import pandas as pd
import tarfile
import urllib.request
import os

print("Downloading dataset directly from Facebook Servers...")
url = "https://dl.fbaipublicfiles.com/parlai/empatheticdialogues/empatheticdialogues.tar.gz"

if not os.path.exists("empatheticdialogues.tar.gz"):
    urllib.request.urlretrieve(url, "empatheticdialogues.tar.gz")
    with tarfile.open("empatheticdialogues.tar.gz", "r:gz") as tar:
        tar.extractall()

# Load into Pandas dataframe
df = pd.read_csv("empatheticdialogues/train.csv", on_bad_lines='skip')
print(f"\nDataset size: {len(df)} total dialogue lines")

print("\n--- Example Line ---")
print("Emotion:", df.iloc[0]['context'])
print("Text:", df.iloc[0]['utterance'])

/tmp/ipykernel_55/1965798728.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()



Dataset size: 76668 total dialogue lines

--- Example Line ---
Emotion: sentimental
Text: I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we felt like the only people in the world.


In [8]:
print("Structuring conversations...")
df = df.sort_values(['conv_id', 'utterance_idx'])

training_strings = []

# Group conversations. We use 1,500 pairs for a quick 2-minute test run. 
# You can remove '[:1500]' to safely train on the entire dataset if you wish!
grouped = list(df.groupby('conv_id'))[:2000] 

for conv_id, group in grouped:
    utterances = group['utterance'].tolist()
    # Combine every two utterances into a User/Therapist pair
    for i in range(len(utterances) - 1):
        user_text = utterances[i].replace('_comma_', ',')
        bot_text = utterances[i+1].replace('_comma_', ',')
        
        # Format: <|user|> I feel sad <|bot|> I am here for you <|endoftext|>
        text = f"<|user|> {user_text} <|bot|> {bot_text} <|endoftext|>"
        training_strings.append(text)

print(f"Created {len(training_strings)} training pairs.")

from datasets import Dataset
train_dataset = Dataset.from_dict({"text": training_strings})

Structuring conversations...
Created 7057 training pairs.


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "distilgpt2"
print(f"Loading {MODEL_ID}...")

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. Add padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Add custom tags
special_tokens_dict = {'additional_special_tokens': ['<|user|>', '<|bot|>']}
tokenizer.add_special_tokens(special_tokens_dict)

# 4. Load Model straight into the Kaggle GPU!
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")

# 5. Increase vocabulary limit
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.eos_token_id

print("Model and tokenizer ready!")

Loading distilgpt2...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and tokenizer ready!


In [10]:
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# NEW SFTConfig - optimized for GPU!
training_args = SFTConfig(
    output_dir="./results",                   
    per_device_train_batch_size=4,  
    learning_rate=2e-5,             
    num_train_epochs=2,             
    optim="adamw_torch",            
    logging_steps=50,               
    report_to="none",               
    dataset_text_field="text"       # Note: We removed 'use_cpu=True' because Kaggle uses GPU!
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    data_collator=collator,
)

print("Trainer initialized! Ready to train.")

Adding EOS to train dataset:   0%|          | 0/7057 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7057 [00:00<?, ? examples/s]

Trainer initialized! Ready to train.


In [11]:
import time

print("Starting Fine-tuning on GPU!")
start_time = time.time()

trainer.train()

end_time = time.time()
print(f"Training complete in {(end_time - start_time) / 60:.1f} minutes.")

OUTPUT_DIR = "mental_health_bot_model"
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to Kaggle storage!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.


Starting Fine-tuning on GPU!


Step,Training Loss
50,3.304196
100,2.924395
150,2.883163
200,2.801566
250,2.848203
300,2.763609
350,2.788212
400,2.731625
450,2.729396
500,2.762332


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete in 5.2 minutes.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to Kaggle storage!


In [12]:
from transformers import pipeline

print("Loading our new fine-tuned model...")
bot_pipe = pipeline(
    "text-generation", 
    model="mental_health_bot_model", 
    tokenizer="mental_health_bot_model",
    device=0 # Uses Kaggle's primary GPU for ultra-fast text generation
)

def chat_with_bot(patient_message):
    print("\n" + "="*50)
    print("PATIENT:", patient_message)
    print("="*50)
    
    # 1. Simple Safety Check
    crisis_keywords = ["suicide", "kill myself", "end my life"]
    if any(k in patient_message.lower() for k in crisis_keywords):
        print("BOT: I hear your pain. You are not alone. Please call an emergency hotline right now.")
        return
        
    # 2. Format the prompt
    formatted_prompt = f"<|user|> {patient_message} <|bot|>"
    
    # 3. Flaw Fixes: Repetition penalty helps prevent weird infinite loops
    outputs = bot_pipe(
        formatted_prompt, 
        max_new_tokens=60, 
        do_sample=True, 
        temperature=0.7, 
        repetition_penalty=1.2,          
        pad_token_id=tokenizer.eos_token_id, 
        eos_token_id=tokenizer.eos_token_id,
        truncation=True
    )
    
    # Extract only the newly generated bot text
    generated_text = outputs[0]['generated_text']
    response = generated_text.split("<|bot|>")[-1].strip()
    
    print("BOT:", response)

# Run test prompts
chat_with_bot("I just feel so overwhelmed with work lately.")
chat_with_bot("Nobody understands how hard this is for me.")

Loading our new fine-tuned model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'eos_token_id', 'temperature', 'do_sample', 'repetition_penalty', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PATIENT: I just feel so overwhelmed with work lately.


Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BOT: Congratulations! That's great, that is the best part about your dedication and accomplishment :) You're a talented person who works hard to achieve things at home without being afraid or distracted by others such as yourself., but you have faith in

PATIENT: Nobody understands how hard this is for me.
BOT: i am proud of myself im so happy my family was able to get everything they needed I wanted in life that would have been great! <3 You must be very sad today, because you still don't want something like it anymore :) -Pray all your children and their adult grandchildren before too long


In [16]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(hf_token)
    
    repo_name = "mental-health-support-bot" # Change this if you want
    print(f"Pushing to Hugging Face Hub as {repo_name}...")
    
    # Push the model and tokenizer
    model.push_to_hub(repo_name)
    tokenizer.push_to_hub(repo_name)
    print("Success! Your model is live!")
except Exception as e:
    print("Skipping Hugging Face push. Setup your HF_TOKEN in Kaggle Secrets if you want to push!", e)

Pushing to Hugging Face Hub as mental-health-support-bot...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Success! Your model is live!
